# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Appendix A: *Constructing a Mesh Network*
##### Version Number: 3.0
---
### Contents  
> 1. *ArcGIS workflow* 
> 2. *Visualization*
---
### Notes

- ArcGIS workflow for constructing a mesh network of sampling points equally spaced throughout California.
---
### Inputs
- `California shape file` - California boundaries
- `elevation raster` - Raster to calculate elevation for sampling points
- `CDFW regions` - California defined regions that offer different climate conditions
- `WUI` - Wildland urban interface data from California State Geoportal
- `US 2020 Census Block Data` - From California State Geoportal
- `Eco Regions` - California Eco Regions for California State Geoportal
---
### Outputs  
- `sampling_points.csv` - completed mesh network 
---
### User Created Dependencies  
---

---
### Third Party Dependencies

In [ ]:
import pandas as pd

### ArcGIS Pro Workflow — Building the Mesh Network

*Projection Used: CRS: NAD 1983 California Statewide Lambert (meters)*

#### 1. Load sampling points, weather stations, and elevation raster
Inputs: `California shape file`, `elevation raster`, `CDFW regions`\
Tool: Add layers via Catalog → Folders/Databases → Add to Map\

#### 2. Create mesh network clipped to california
Input: `California shape file` layer\
Tool: Analysis → Tools → Create Fishnet\
Output: `Sampling_Points` layer
> Notes: 
> - cell height and width = 0.5
> - Creates equally spaced samples 46 Km apart throughout California

#### 3. Extract latitude and longitude into fields
Input: `Sampling_Points` layer\
Tool: Analysis → Tools → Calculate Geometry Attributes\
Output: `Sampling_Points` with Latitude and Longitude Fields
> Notes: Projection used WGS 1984 Web Mercator (Auxiliary Sphere)

#### 4. Assign region ID to each sampling point
Input: `Sampling_points` and `CDFW_regions` layers\
Tool: Analysis → Tools → Overlay → Spatial Join\
Output: `Sampling_Points` with Region_ID and Region_Name fields
> Notes:
> - Target Features: sampling points
> - Join Features: CDFW regions

#### 5. Extract elevation values to points
Input: `Sampling_Points` and `Elevation` Raster\
Tool: Analysis → Tools → Spatial Analyst Tools → Extraction → Extract Values to Points\
Output: `Sampling_Points_with_Elevation`
> Notes: Adds a new numeric field like Elevation to each point.



#### 6. Create buffer around points
Input: `Sampling_points_with_Elevation`\
Tool: Analysis → Tools → Buffer\
Output: `Sampling_Points_Buffered`
> Notes:
> - Buffer size is 36 Km
> - This ensures complete coverage of California but does result in minor overlap

#### 7. Add and Calculate area of WUI regions
Input: `WUI`\
Tool: Calculate Geometry Atrributes\
Output: `WUI`
> Notes: 
> - Adds area field in square meters

#### 8. Intersect WUI layer with buffer layer
Input: `Sampling_Points_Buffered`, `WUI`\
Tool: Analysis → Tools → Intersect\
Output: `Sampling_Points_with_WUI`
> Notes: 

#### 9. Calculate WUI area within buffered radius of sampling points
Input: `Sampling_Points_with_WUI`\
Tool: Analysis → Tools → Summarize Within\
Output: `Sampling_Points_with_WUI`
> Notes: Perform three times once for each WUI zone
> - Interface, Intermix, Influence
> - Adds three new fields containing the total area each zone in a buffer

#### 10. Attach summed areas to sampling points
Input: `Sampling_Points_with_WUI`, `Sampling_points_with_Stations`\
Tools: Analysis → Tools → Spatial Join\
Output: `Sampling_Points_WUI`
> Notes:
> - Target Features: your sampling points
> - Join Features: your buffer polygons with SUM_Clipped_Area
> - Join Operation: JOIN_ONE_TO_ONE (since one point is in one buffer)
> - Match Option: INTERSECT
> - Joins the three summed area fields with the point in the center of each buffer

#### 11. Summarize census info
Input: `Buffered_Sampling_Points`,`2020_Census_Block`\
Tool: Analysis → Tools → Summarize Within \
Output: `Buffered Sampling_Points_with_sums`
> Notes: 
> - Sum population, housing and area fields within each buffered area
> - Adds three new fields containing the total area each zone in a buffer

#### 12. Attach summed census info to sampling points
Input: `Sampling_Points_WUI`, `Buffered Sampling_Points_with_sums`\
Tools: Analysis → Tools → Spatial Join\
Output: `Sampling_Points_Pop`
> Notes:
> - Target Features: your sampling points with WUI
> - Join Features: your buffer polygons with SUM_Clipped_Area
> - Join Operation: JOIN_ONE_TO_ONE (since one point is in one buffer)
> - Match Option: INTERSECT
> - Joins the three summed area fields with the point in the center of each buffer

#### 13. Attach ecoregions to sampling points
Input: `Sampling_Points_Pop`, `California_Eco_Regions`\
Tools: Analysis → Tools → Spatial Join\
Output: `Sampling_Points_Eco`
> Notes:
> - Target Features: your sampling points
> - Join Features: California Eco Regions
> - Join Operation: JOIN_ONE_TO_ONE
> - Match Option: Within
> - Joins each point with the eco region it is in

#### 16. Join lookup table
Match Region Code to Region Name and a unique numeric ID for models

In [4]:
ecoregion_meta = pd.read_csv('../data/raw/ecoregion_meta.csv')
ecoregion_meta

,Eco_Code,Eco_Name,Eco_ID
0,261,California Coastal Chaparral Forest and Shrub,1
1,262,California Dry Steppe,2
2,263,California Coastal Steppe-Mixed Forest-Redwood...,3
3,322,American Semi-Desert and Desert,4
4,341,Intermountain Semi-Desert and Desert,5
5,342,Intermountain Semi-Desert,6
6,M242,Cascade Mixed-Forest-Coniferous Forest-Alpine ...,7
7,M260,Mediterranean Regime Mtns,8
8,M261,Sierran Steppe-Mixed Forest-Coniferous Forest-...,9
9,M262,California Coastal Range Open Woodland-Shrub-C...,10



#### 13. Export the final mesh layer
Tool: Data → Export to Table

<img src="../data/maps/mesh.jpg" width="800">

#### Final Product 

A csv of a complete mesh layer that combines:
- Sampling points spaced 46 km apart
- Latitude and Longitude of each point
- Raster elevation values
- Population statistics in buffered radius
- Region attributes (CDFW region and Eco Regions)
- *Interface*, *Intermix*, *Influence* zones areas summed up in a 36Km Buffer Radius 